# Modelado Secuencial y Extracción de Características para el Reconocimiento de Lengua de Señas Argentina utilizando el Dataset LSA64

Trabajo realizado por el grupo 8:

*   Younes Aghani
*   Álvaro Gómez García
*   Carlos García Jiménez
*   Alonso Lucas Juberías

El reconocimiento automático de lengua de señas es un área clave en la intersección de la inteligencia artificial y la accesibilidad, enfocada en romper barreras de comunicación para personas con discapacidad auditiva. En este proyecto, se aborda el reconocimiento de la Lengua de Señas Argentina (LSA) utilizando el dataset LSA64 ([LSA64: A Dataset for Argentinian Sign Language](https://facundoq.github.io/datasets/lsa64/)), un conjunto de datos ampliamente reconocido que contiene videos de 64 señas distintas realizadas por diferentes participantes. La metodología propuesta combina técnicas de extracción de características mediante redes convolucionales preentrenadas, como ResNet50, con modelos secuenciales basados en GRU para capturar información temporal. El objetivo es desarrollar un modelo robusto y eficiente para el dataset completo.

Especificamente se ha utilizado el siguiente [dataset de kaggle](https://www.kaggle.com/datasets/marcmarais/lsa64-videos) donde se incorpora los csv con las etiquetas correspondientes a cada video. Disponemos pues de 3200 videos para las 64 clases de las que  analizaremos posteriormente la distribución de estas, aunque nos aseguran que es uniforme. En el mismo dataset de kaggle nos dan de partida una partición de train(2560)/test(640) que pareciendonos esta una buena distribución, no modificaremos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys,os

In [ ]:
!pip install git+https://github.com/tensorflow/docs

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras import models, layers
import matplotlib.pyplot as plt

### Carga y analisis del Dataset LSA64

Comenzamos especificando las rutas de nuestros datos por separado, tanto de los videos de traint y test con sus respectivas etiquetas en los csv. Finalmente cargamos las etiquetas utilizando pandas. Es importante mencionar que se debe indicar a la función read_csv que keep_default_na=False debido a que una de nuestras etiquetas es "None" pero porque el signo que se representa es el de "None", no porqué este no tenga etiqueta.

In [ ]:
# Ruta principal
base_path = "/content/drive/My Drive/Colab Notebooks"

# Rutas a los conjuntos de datos y videos (https://www.kaggle.com/datasets/marcmarais/lsa64-videos)
train_csv = base_path + "/lsa64_kaggle/Signer_Independent_data/Signer_Independent_data/LSA64_videos_signer_independent_train.csv"
test_csv = base_path + "/lsa64_kaggle/Signer_Independent_data/Signer_Independent_data/LSA64_videos_signer_independent_test.csv"
train_videos = base_path + "/lsa64_kaggle/Signer_Independent_data/Signer_Independent_data/Signer_Independent_Splits/train"
test_videos = base_path + "/lsa64_kaggle/Signer_Independent_data/Signer_Independent_data/Signer_Independent_Splits/test"

# Carga de los datos asegurando que "None" no se interprete como NaN
train_df = pd.read_csv(train_csv, na_values=[], keep_default_na=False, index_col=0)
test_df = pd.read_csv(test_csv, na_values=[], keep_default_na=False, index_col=0)

Continuamos mostrando como estan estructurados los ficheros csv que hemos cargado, teniendo estos una linea para cada video, 2560 para train y 640 para test. En cada linea (dato del csv) vemos el nombre del video y la etiqueta real que posee. Aunque el número total de videos parece razonable, cuando se considera el tamaño del espacio de búsqueda (64 clases), el conjunto de datos podría ser limitado.

In [ ]:
# Impresión de estadísticas básicas y contenido de los datos
print(f"Total videos for training: {len(train_df)}")  # Imprime el número total de videos de entrenamiento
print(f"Total videos for testing: {len(test_df)}")  # Imprime el número total de videos de prueba
print(train_df)  # Muestra el contenido del DataFrame de entrenamiento
print(test_df)  # Muestra el contenido del DataFrame de prueba

Continuamos mostrando todas las clases del data set que estamos usando para comprobar que este está completo y que coindice con los de test. Vemos una pequeña diferencia que se ha de aclarar con respecto del data set oficial, y es que Pink aquí lo llamaremos Red2 (videos 008_).

Contamos entonces con 64 clases en nuestros conjuntos de datos.

In [ ]:
# Verificar el número total de clases únicas en el conjunto de entrenamiento
print("Clases en el conjunto de entrenamiento:")
print(train_df['class_name'].unique())
print("\nNúmero total de clases en el conjunto de entrenamiento: ",len(train_df['class_name'].unique()))

# Verificar el número total de clases únicas en el conjunto de prueba
print("\nClases en el conjunto de prueba:")
print(test_df['class_name'].unique())
print("\nNúmero total de clases en el conjunto de test: ",len(test_df['class_name'].unique()))

Ahora pasamos a comprobar la distribución de los datos sobre las clases, para ello simplemente contamos el número de videos por clase.

In [ ]:
# Estadísticas básicas
train_stats = train_df['class_name'].value_counts()  # Conteo de videos por clase (entrenamiento)
test_stats = test_df['class_name'].value_counts()    # Conteo de videos por clase (prueba)

# Mostrar las estadísticas como tablas
print("Estadísticas del conjunto de entrenamiento:")
print(train_stats)

print("\nEstadísticas del conjunto de prueba:")
print(test_stats)

Para una visualización más clara y sencilla, creamos una gráfica y podemos ver facilmente como para todas las clases existen 40 videos en el conjunto de entrenamiento y 10 videos en el conjunto de test. Aunque tener 40 ejemplos por clase en el entrenamiento es útil para la clasificación inicial, puede ser insuficiente para modelos complejos, especialmente si las clases tienen gestos similares o alta variabilidad.En definitiva, aunque el dataset es balanceado, su tamaño relativamente pequeño y el bajo número de ejemplos por clase podrían limitar el rendimiento de un modelo más complejo.

In [ ]:
plt.figure(figsize=(13, 6))  # Ajustar el tamaño de la figura para más espacio horizontal
train_stats.plot(kind="bar", color="skyblue", alpha=0.6, label="Train Data")
test_stats.plot(kind="bar", color="orange", alpha=0.6, label="Test Data")
plt.title("Comparación de Distribución de Clases en Entrenamiento y Prueba")
plt.xlabel("Clase")
plt.ylabel("Número de videos")
plt.xticks(rotation=90, ha="center")  # Rotar etiquetas a 90 grados y centrar alineación horizontal
plt.legend()
plt.tight_layout()  # Ajustar márgenes para que las etiquetas no se corten
plt.show()

Con intención de mostrar un ejemplo de nuestro conjunto de datos (no siendo muy fiable mostrarlo como video en un google notebook), la siguiente función permite visualizar una selección de 12 fotogramas equidistantes de un video especificado por su ruta. Primero, verifica si la ruta del archivo de video es válida y, en caso contrario, muestra un mensaje de error. Luego, utiliza OpenCV para abrir el video y calcular los índices correspondientes a los 12 fotogramas equidistantes a lo largo de su duración. A continuación, recorre los fotogramas del video y selecciona aquellos que coinciden con los índices calculados, convirtiéndolos al formato RGB para una visualización adecuada. Los fotogramas capturados se almacenan en una lista. Después, se crea una cuadrícula de 2 filas por 6 columnas utilizando Matplotlib para mostrar los fotogramas seleccionados. Los ejes de las imágenes se ocultan para una visualización más limpia, y se ajusta el diseño de la cuadrícula para evitar solapamientos. Finalmente, la figura generada se muestra y se cierra para liberar memoria.

In [ ]:
def display_video_frames(video_path):
    # Verifica si la ruta del video es válida
    if not os.path.exists(video_path):
        print("El archivo de video no se encuentra en la ruta especificada.")
        return

    # Abre el video utilizando OpenCV
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Número total de fotogramas del video

    # Calcula los índices de los 12 fotogramas equidistantes
    num_frames_to_capture = 12
    frame_indices = [int(i * (total_frames - 1) / (num_frames_to_capture - 1)) for i in range(num_frames_to_capture)]

    frames = []  # Lista para almacenar los fotogramas seleccionados
    current_index = 0  # Índice actual de la lista de frame_indices

    for i in range(total_frames):
        ret, frame = cap.read()  # Lee un fotograma del video
        if not ret:  # Si no se pudo leer, detiene el procesamiento
            break

        if i == frame_indices[current_index]:  # Captura el fotograma si es uno de los índices calculados
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Convierte a RGB
            frames.append(frame_rgb)  # Agrega el fotograma convertido a la lista
            current_index += 1  # Pasa al siguiente índice

        if current_index >= len(frame_indices):  # Detiene si se han capturado todos los fotogramas
            break

    cap.release()  # Libera el video una vez que se hayan procesado los fotogramas

    # Crea una cuadrícula para mostrar los 12 fotogramas
    fig, axes = plt.subplots(2, 6, figsize=(15, 5))  # 2 filas x 6 columnas
    for i, ax in enumerate(axes.flatten()):  # Recorre los ejes de la cuadrícula
        if i < len(frames):  # Muestra un fotograma si existe
            ax.imshow(frames[i])
            ax.axis('off')  # Oculta los ejes
        else:
            ax.axis('off')  # Oculta los ejes vacíos en caso de que haya menos de 12 fotogramas

    plt.tight_layout()  # Ajusta el diseño de la cuadrícula
    plt.show()  # Muestra los fotogramas
    plt.close(fig)  # Cierra la figura para liberar memoria

Por tanto haciendo uso de la función anterior, visualicemos el primer video en fotogramas para tener una primera idea del problema al que nos enfrentamos. Este video pertenece a la clase 'Opaque' y se puede ver como hay una persona de cintura para arriba realizando el gesto correspondiente del lenguaje de signos de esta clase en fotogramas. Observamos que la persona realiza los gestos con un fondo simple y uniforme, lo que reduce posibles distracciones por elementos externos. Además, el uso de un guante de color brillante (en este caso, rojo) en la mano activa facilita la segmentación y seguimiento de los movimientos, lo que podría beneficiar el modelo al resaltar las partes más relevantes de la acción. Los gestos presentan variaciones sutiles en la posición de la mano, por lo que el modelo debe capturar detalles temporales y espaciales con precisión. También es evidente que las expresiones faciales y el torso permanecen estáticos, lo que sugiere que estas áreas no aportarán información significativa para el reconocimiento y podrían omitirse durante la extracción de características. Este contexto inicial nos sugiere que un modelo basado en redes neuronales recurrentes o transformadores, combinado con un extractor de características visuales preentrenado, será esencial para captar la dinámica temporal y las características clave de los gestos.

In [ ]:
# Mostramos el primer video del conjunto de train en 12 frames equidistantes
video_ejemplo1 = train_videos + "/001_001_001.mp4"
display_video_frames(video_ejemplo1)

En este segundo video (gesto 'To Land') observamos que se mantienen ciertas características consistentes con el primer ejemplo, como el uso de guantes de colores brillantes (rojo y verde) para resaltar las manos que realizan el gesto, y un fondo simple y uniforme que minimiza las distracciones. Sin embargo, también notamos nuevas consideraciones importantes. Primero, el gesto involucra movimientos más complejos que incluyen ambas manos, lo que podría requerir un modelo capaz de capturar relaciones espaciales y temporales entre las manos. Segundo, la iluminación parece diferente, lo que podría afectar la calidad de la segmentación y ser un factor a considerar para garantizar la robustez del modelo ante variaciones lumínicas. Además, el gesto incluye posiciones específicas de los dedos y movimientos relativos entre las manos, lo que sugiere que el modelo deberá ser particularmente sensible a detalles finos. En general, este video refuerza la necesidad de un modelo robusto que combine una buena extracción de características visuales y un análisis temporal preciso para reconocer gestos en diversas condiciones.

In [ ]:
# Mostramos otro video del conjunto de train en 12 frames equidistantes
video_ejemplo54 = train_videos + "/054_001_001.mp4"
display_video_frames(video_ejemplo54)

### Preprocesado de videos

Antes de comenzar con el preprocesado de videos quería resaltar que vamos ha utilizar keras.


Keras es una biblioteca de alto nivel para el desarrollo de modelos de aprendizaje profundo que proporciona una interfaz sencilla y modular, integrada de forma nativa en TensorFlow. Su principal objetivo es facilitar la construcción y entrenamiento de redes neuronales mediante una API intuitiva. Keras soporta la creación de arquitecturas de modelos secuenciales, funcionales y personalizados, lo que la hace muy flexible para resolver una amplia gama de problemas de aprendizaje profundo. En este proyecto de reconocimiento de gestos en lenguaje de señas, Keras es particularmente útil porque ofrece herramientas avanzadas para el procesamiento de datos (como StringLookup para etiquetas), preprocesamiento de imágenes y la integración de modelos preentrenados, como ResNet50, que optimizan la extracción de características visuales.

Además, Keras es altamente compatible con TensorFlow, permitiendo el uso de GPU para acelerar el entrenamiento y optimizar el rendimiento, algo crucial para problemas que manejan datos de alta dimensionalidad, como videos. Las capacidades de Keras para manejar datos secuenciales y arquitecturas recurrentes, como GRU o LSTM, son ideales para capturar la dinámica temporal de los gestos.

En el siguisiente código se divide los datos del conjunto de entrenamiento en dos subconjuntos: uno para entrenamiento (train_df) y otro para validación (val_df). Esto se realiza mediante la función train_test_split de sklearn, que asegura que el 20% de los datos originales se asignen al conjunto de validación. La división utiliza la columna class_name para garantizar que la distribución de clases sea proporcional en ambos subconjuntos (estratificación). La semilla de aleatoriedad (random_state=42) asegura que la división sea reproducible en ejecuciones futuras. Finalmente, se restablecen los índices de ambos DataFrames con reset_index para que las filas estén ordenadas consecutivamente, eliminando los índices anteriores.

Es crucial separar un conjunto de validación antes de la extracción de características para evitar el fenómeno de data snooping, que ocurre cuando información del conjunto de validación contamina el modelo durante el entrenamiento. Si las características se extraen conjuntamente, el modelo podría ajustarse inadvertidamente al conjunto de validación, llevando a un rendimiento artificialmente elevado y poco representativo en evaluaciones posteriores. Por tanto, esta separación previa asegura que el modelo se evalúe de manera imparcial y que los datos de validación reflejen su desempeño real.

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir los datos del DataFrame original
train_df, val_df = train_test_split(
    train_df,  # DataFrame original
    test_size=0.2,  # 20% para validación
    stratify=train_df["class_name"].values,  # Mantener la proporción de clases
    random_state=42  # Para reproducibilidad
)

# Resetear índices después de la división
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

Finalmente tenemos un conjunto de entrenamiento de 2048 videos y un conjunto de validación de 512 videos.

In [ ]:
# Impresión de estadísticas básicas y contenido de los datos
print(f"Total videos for training: {len(train_df)}")  # Imprime el número total de videos de entrenamiento
print(f"Total videos for validation: {len(val_df)}")  # Imprime el número total de videos de prueba
print(train_df)  # Muestra el contenido del DataFrame de entrenamiento
print(val_df)  # Muestra el contenido del DataFrame de prueba


El siguiente fragmento de código crea un procesador de etiquetas (label_processor) que convierte las clases textuales del conjunto de datos en índices numéricos. Primero, se obtiene el vocabulario único de las clases del conjunto de entrenamiento, ordenado alfabéticamente, para garantizar un mapeo consistente entre texto e índices. Luego, se utiliza keras.layers.StringLookup, que mapea cada clase textual a un número entero, lo que es esencial para que los modelos de aprendizaje profundo procesen las etiquetas de manera eficiente.

In [ ]:
# Crear el vocabulario único basado en las clases del conjunto de entrenamiento
vocabulario = sorted(train_df["class_name"].unique())

# Crear un único label_processor global que será usado en todas las funciones
label_processor = keras.layers.StringLookup(
    num_oov_indices=0,  # No permitir índices para valores fuera del vocabulario
    vocabulary=vocabulario  # Usar el vocabulario derivado del conjunto de entrenamiento
)

A continuación configuraremos la extracción de características, esta es un proceso fundamental en el aprendizaje profundo, donde se identifican patrones o atributos relevantes de los datos que sirven como entrada para modelos más complejos. En este caso, se utiliza un modelo preentrenado, ResNet50, para extraer características visuales de las imágenes de los videos. ResNet50 es una red neuronal convolucional profunda con 50 capas diseñada para resolver tareas de visión por computadora, como clasificación de imágenes. Fue entrenada previamente en el dataset ImageNet, lo que le permite capturar patrones visuales genéricos, como bordes, texturas y formas, que son útiles para una amplia gama de aplicaciones.

Por tanto el siguiente codigo configura ResNet50 como un extractor de características al excluir su capa final completamente conectada (include_top=False), que originalmente está diseñada para clasificar las imágenes en las clases de ImageNet. En su lugar, se utiliza el promedio global de las activaciones (pooling="avg") para generar un vector compacto que representa la información clave de cada imagen. Las imágenes se redimensionan a 128x128 píxeles y pasan por un preprocesamiento específico de ResNet50 (preprocess_input) para normalizar sus valores según lo esperado por la red. Se congelan los pesos de ResNet50 (base_model.trainable = False), evitando que estos sean actualizados durante el entrenamiento, ya que están optimizados para detectar patrones visuales genéricos. Esto ahorra tiempo y recursos computacionales, enfocando el entrenamiento en las capas específicas del modelo final que procesarán estas características para resolver el problema de reconocimiento de gestos.

In [ ]:
IMG_SIZE = 128  # Dimensión de las imágenes (se redimensionarán a 128x128 píxeles)

def build_feature_extractor():
    # Construye un extractor de características basado en ResNet50 preentrenado en ImageNet.
    # Ajusta la configuración de pesos y arquitectura según sea necesario.

    base_model = keras.applications.ResNet50(
        weights="imagenet",      # Cargar pesos preentrenados en ImageNet
        include_top=False,       # Excluir la capa completamente conectada final
        pooling="avg",           # 'average pooling' para las características
        input_shape=(IMG_SIZE, IMG_SIZE, 3),  # Tamaño de entrada (imagen de 128x128 con 3 canales)
    )

    preprocess_input = keras.applications.resnet50.preprocess_input  # Preprocesar imágenes para ResNet50
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))  # Definir los inputs del modelo
    x = preprocess_input(inputs)  # Aplicar preprocesamiento a las imágenes de entrada
    outputs = base_model(x)  # Pasar las entradas por la base del modelo ResNet50

    # Congelar los pesos del modelo base para que no se entrenen
    base_model.trainable = False

    return keras.Model(inputs, outputs, name="feature_extractor")  # Retornar el modelo extractor de características

El siguiente código define una función llamada crop_center_square que recorta un fotograma para obtener un área cuadrada centrada en la imagen. Primero, calcula las dimensiones del fotograma y determina el lado mínimo (entre alto y ancho) para que el recorte sea un cuadrado. Luego, calcula las coordenadas de inicio para centrar el cuadrado y recorta la región correspondiente. Esta operación es útil en nuestro problema porque garantiza que las imágenes de entrada tengan proporciones consistentes, eliminando bordes innecesarios y enfocándose en la región central, donde generalmente ocurren los gestos. Esto mejora la uniformidad de los datos y ayuda al modelo a aprender patrones más relevantes.

In [ ]:
def crop_center_square(frame):
    # Recorta el fotograma para obtener un cuadrado centrado.

    y, x = frame.shape[0:2]  # Obtener las dimensiones del fotograma (alto y ancho)
    min_dim = min(y, x)  # Determinar la dimensión mínima entre alto y ancho
    start_x = (x // 2) - (min_dim // 2)  # Calcular el punto de inicio en el eje x para centrar
    start_y = (y // 2) - (min_dim // 2)  # Calcular el punto de inicio en el eje y para centrar

    # Recortar el fotograma al cuadrado centrado
    return frame[start_y : start_y + min_dim, start_x : start_x + min_dim]

La siguiente función se encarga de procesar un video cargando una cantidad fija de fotogramas (máximo max_frames), seleccionados equitativamente a lo largo de su duración. Utiliza OpenCV para abrir el video y calcula los índices de los fotogramas que deben capturarse, distribuidos uniformemente mediante la función np.linspace. A continuación, recorre todos los fotogramas del video y captura únicamente aquellos en los índices seleccionados. Para cada fotograma, aplica varias transformaciones: recorta la región central para obtener un área cuadrada (con la función crop_center_square), redimensiona el fotograma a un tamaño específico (resize), y convierte su formato de color de BGR (predeterminado en OpenCV) a RGB, que es más adecuado para trabajar con modelos de aprendizaje profundo. Los fotogramas procesados se almacenan en una lista y finalmente se devuelven como un array de NumPy.

El uso de fotogramas para este problema es fundamental porque los videos son secuencias de imágenes que capturan la evolución temporal de los gestos en lenguaje de señas. Seleccionar un número fijo de fotogramas equidistantes ayuda a reducir la cantidad de datos procesados, haciéndolo manejable para el modelo, mientras se preserva la información temporal clave. Además, el recorte y redimensionamiento garantizan que todos los videos tengan una resolución y proporciones consistentes, lo que facilita el entrenamiento del modelo. Este enfoque permite capturar los patrones espaciales (posición de las manos) y temporales (movimientos a lo largo del tiempo), que son esenciales para reconocer correctamente los gestos en este tipo de problemas.

In [ ]:
def load_video(path, max_frames=20, resize=(IMG_SIZE, IMG_SIZE)):
    # Carga un video desde la ruta, recorta los fotogramas, redimensiona y selecciona un número fijo de frames equidistantes.

    cap = cv2.VideoCapture(path)  # Abrir el archivo de video
    frames = []  # Lista para almacenar los fotogramas procesados

    try:
        # Obtener el número total de fotogramas del video
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        # Generar índices equidistantes para los fotogramas a capturar
        frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=np.int32)

        for idx in range(total_frames):  # Iterar sobre todos los fotogramas del video
            ret, frame = cap.read()  # Leer un fotograma
            if not ret:  # Si no se puede leer el fotograma, salir del bucle
                break

            # Procesar solo los fotogramas seleccionados por los índices
            if idx in frame_indices:
                # Recortar el fotograma al cuadrado central
                frame = crop_center_square(frame)
                # Redimensionar el fotograma al tamaño especificado
                frame = cv2.resize(frame, resize)
                # Convertir el fotograma de formato BGR a RGB
                frame = frame[:, :, [2, 1, 0]]
                frames.append(frame)  # Agregar el fotograma procesado a la lista

                # Salir si se alcanzó el número deseado de fotogramas
                if len(frames) == max_frames:
                    break
    finally:
        cap.release()  # Liberar el archivo de video después de procesarlo

    return np.array(frames)  # Retornar los fotogramas procesados como un array de NumPy

Este código realiza data augmentation para enriquecer los datos de entrada y aumentar la robustez del modelo de reconocimiento de señas. Los siguientes pasos de augmentación se implementan:

1. Ajuste de brillo: Aplica una variación aleatoria en el brillo de cada fotograma, simulando condiciones de iluminación cambiantes, como una sala más luminosa o más oscura. Esto mejora la capacidad del modelo para generalizar bajo diferentes condiciones de luz.

2. Ajuste de contraste: Modifica el contraste dentro de un rango moderado, ayudando al modelo a manejar mejor las variaciones en la calidad visual de los videos.

3. Desplazamiento pequeño: Introduce un desplazamiento aleatorio en las direcciones horizontal y vertical, simulando escenarios donde la persona que realiza las señas no está perfectamente centrada en el video.

4. Zoom Central: Recorta y redimensiona el fotograma, emulando diferentes distancias de la cámara respecto a la persona. Esto permite que el modelo sea más robusto frente a cambios de escala.

5. Ruido Aleatorio: Agrega ruido gaussiano para imitar distorsiones comunes en grabaciones, como el ruido de sensores de cámaras o compresión de video.

6. Validación de valores: Se asegura que los valores de los píxeles estén dentro de un rango válido (0-255), lo que previene artefactos debido a operaciones matemáticas durante la augmentación.

Estas técnicas son beneficiosas porque aumentan la variabilidad de los datos sin necesidad de recolectar nuevos videos, permitiendo que el modelo sea más robusto frente a cambios en iluminación, posición, distancia, y calidad de grabación.

In [ ]:
import random

def augment_frame(frame):
    # Asegurar que el frame sea un tensor para las operaciones de TensorFlow
    frame = tf.convert_to_tensor(frame, dtype=tf.float32)

    # 1. Ajuste de brillo
    frame = tf.image.random_brightness(frame, max_delta=0.2)  # Variación moderada de brillo

    # 2. Ajuste de contraste
    frame = tf.image.random_contrast(frame, lower=0.8, upper=1.2)  # Ajuste moderado de contraste

    # Convertir el frame nuevamente a NumPy para operaciones con OpenCV
    frame = frame.numpy()

    # 3. Desplazamiento pequeño
    if random.random() > 0.5:  # Probabilidad del 50%
        tx = random.randint(-10, 10)  # Desplazamiento en x
        ty = random.randint(-10, 10)  # Desplazamiento en y
        h, w, c = frame.shape
        M = np.float32([[1, 0, tx], [0, 1, ty]])
        frame = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # 4. Zoom Central (Recorte y redimensionado)
    if random.random() > 0.5:  # Probabilidad del 50%
        scales = [1.0, 0.9, 0.8]  # Escalas de zoom
        scale = random.choice(scales)
        h, w, _ = frame.shape
        crop_h, crop_w = int(h * scale), int(w * scale)
        offset_h, offset_w = (h - crop_h) // 2, (w - crop_w) // 2
        frame = frame[offset_h:offset_h + crop_h, offset_w:offset_w + crop_w, :]
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))  # Redimensionar

    # 5. Ruido Aleatorio
    if random.random() > 0.5:  # Probabilidad del 50%
        noise = np.random.normal(0, 10, frame.shape)  # Ruido gaussiano con std=10
        frame = np.clip(frame + noise, 0, 255)

    # 6. Asegurar valores válidos de píxeles y convertir de nuevo a tensor si es necesario
    frame = tf.clip_by_value(tf.convert_to_tensor(frame, dtype=tf.float32), 0, 255)

    return frame.numpy()  # Retornar como array de NumPy

La siguiente función se encarga de procesar un conjunto de videos y preparar los datos para ser utilizados en un modelo de aprendizaje profundo. Recibe como entrada un DataFrame (df) que contiene información sobre los videos, una ruta raíz (root_dir) donde se encuentran los archivos, y un procesador de etiquetas (label_processor) para convertir las etiquetas textuales en índices numéricos. Para cada video, se extraen características relevantes y se generan máscaras para identificar cuáles fotogramas son válidos dentro de una longitud máxima definida (MAX_SEQ_LENGTH).

La función primero inicializa matrices para almacenar las características de los fotogramas (frame_features) y las máscaras (frame_masks) para todos los videos. Luego, utiliza un bucle para iterar sobre cada video, cargándolo con la función load_video, que extrae un número fijo de fotogramas. Los fotogramas son procesados en lotes y sus características visuales se extraen utilizando el modelo extractor de características (feature_extractor). Estas características, que son vectores numéricos de tamaño fijo (NUM_FEATURES), representan patrones visuales clave en los fotogramas, como posiciones y movimientos de las manos.

Cada video se representa como una secuencia de características, y se aplica una máscara para marcar cuáles fotogramas son válidos (en caso de que el video tenga menos fotogramas que MAX_SEQ_LENGTH). Finalmente, la función devuelve las características extraídas, las máscaras y las etiquetas correspondientes en formato numérico.

A la función se le indica si se desea aplicar data augmention a la preparación de los videos, esto es llamar para cada frame a la función de augment_frame. Aunque como ya hemos mencionado anteriormente esto solo será para el conjunto de entrenamiento.

In [ ]:
from tqdm import tqdm

MAX_SEQ_LENGTH = 50  # Longitud máxima de las secuencias de video
NUM_FEATURES = 2048  # Número de características por video

def prepare_all_videos_with_augmentation(df, root_dir, label_processor, augment=False):
    # Crear el modelo extractor de características
    feature_extractor = build_feature_extractor()
    num_samples = len(df)  # Número total de muestras en el DataFrame
    video_paths = df["video_name"].values.tolist()  # Rutas de los videos

    # Convertir las etiquetas textuales a índices numéricos
    labels = df["class_name"].fillna(label_processor.get_vocabulary()[0]).values
    labels = label_processor(labels[..., None]).numpy()

    # Inicializar matrices para almacenar características y máscaras
    frame_masks = np.zeros(shape=(num_samples, MAX_SEQ_LENGTH), dtype="bool")
    frame_features = np.zeros(
        shape=(num_samples, MAX_SEQ_LENGTH, NUM_FEATURES), dtype="float32"
    )

    print('Obteniendo frames y extrayendo features...')  # Mensaje de inicio

    # Iterar sobre cada video utilizando tqdm para mostrar el progreso
    for idx, path in enumerate(tqdm(video_paths, desc="Procesando videos", unit="video")):
        # Cargar los fotogramas del video
        frames = load_video(os.path.join(root_dir, path), max_frames=20)

        # Aplicar data augmentation si es necesario
        if augment:
            frames = np.array([augment_frame(frame) for frame in frames])

        frames = frames[None, ...]  # Expandir dimensiones para simular un batch de tamaño 1

        # Inicializar características y máscaras temporales para un video
        temp_frame_mask = np.zeros((1, MAX_SEQ_LENGTH), dtype="bool")
        temp_frame_features = np.zeros((1, MAX_SEQ_LENGTH, NUM_FEATURES), dtype="float32")

        # Extraer características para cada fotograma del video cargado
        for i, batch in enumerate(frames):
            video_length = batch.shape[0]  # Número de fotogramas en el video
            length = min(MAX_SEQ_LENGTH, video_length)  # Limitar la longitud al máximo permitido

            # Iterar sobre los fotogramas válidos y extraer sus características
            for j in range(length):
                temp_frame_features[i, j, :] = feature_extractor.predict(
                    batch[None, j, :], verbose=0  # Extraer características con el modelo preentrenado
                )

            # Actualizar la máscara para marcar los fotogramas válidos
            temp_frame_mask[i, :length] = 1

        # Almacenar las características y máscaras procesadas en las matrices principales
        frame_features[idx] = temp_frame_features.squeeze()
        frame_masks[idx] = temp_frame_mask.squeeze()

    # Retornar las características, máscaras y etiquetas correspondientes
    return (frame_features, frame_masks), labels

Utilizamos la función anterior para el conjunto de entrenamiento y validación

In [ ]:
val_data, val_labels = prepare_all_videos_with_augmentation(val_df, train_videos, label_processor, False)

In [ ]:
train_data, train_labels = prepare_all_videos_with_augmentation(train_df, train_videos, label_processor, True)

Mediante el siguiente codigo podemos ver un ejemplo de como se carga un video y los frames resultates de recortar y coger aquellos equidistantes, siendo estos frames los que crea y utiliza la función de prepare_all_videos.

In [ ]:
def visualize_processed_frames(video_path, max_frames=20):
    # Cargar los frames procesados del video
    frames = load_video(video_path, max_frames=max_frames)

    # Configurar el tamaño de la cuadrícula para 20 frames
    rows = 4
    cols = 5

    # Crear una cuadrícula para mostrar los frames
    fig, axes = plt.subplots(rows, cols, figsize=(15, 10))
    fig.suptitle("Ejemplo de frames procesados", fontsize=16)

    for i, ax in enumerate(axes.flatten()):  # Recorre los ejes de la cuadrícula
        if i < len(frames):  # Muestra un frame si existe
            ax.imshow(frames[i] / 255.0)  # Normalizar valores de píxeles a [0, 1]
            ax.axis('off')  # Oculta los ejes
        else:
            ax.axis('off')  # Oculta celdas vacías

    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Ajusta el diseño
    plt.show()

In [ ]:
video_ejemplo32 = train_videos + "/032_001_001.mp4"
visualize_processed_frames(video_ejemplo32)

### Creación del modelo

Para la creación del modelo se ha utilizado GRU, pero para entenderlo primero se debe explicar que son las redes neuronales recurrentes (RNN), estas son un tipo de red diseñado para procesar datos secuenciales, como texto, videos o series temporales. A diferencia de las redes tradicionales, las RNN tienen conexiones cíclicas que permiten que la información persista a lo largo del tiempo, lo que las hace ideales para capturar dependencias temporales en los datos. En cada paso, las RNN procesan un elemento de la secuencia y actualizan un estado interno, que actúa como una memoria de lo que ha ocurrido anteriormente. Sin embargo, las RNN estándar tienen problemas para aprender dependencias a largo plazo debido al problema del desvanecimiento o explosión del gradiente. Este problema se aborda con variantes como GRU y LSTM, que introducen mecanismos para gestionar mejor la memoria y las dependencias temporales.

Las GRU (Gated Recurrent Units) son una variante de las RNN que simplifican el diseño de las LSTM (Long Short-Term Memory). Las GRU tienen dos puertas principales: una puerta de actualización y una puerta de reinicio. La puerta de actualización controla cuánto de la memoria pasada debe mantenerse, mientras que la puerta de reinicio decide cuánta información pasada debe descartarse. A diferencia de las LSTM, las GRU no tienen una celda de memoria separada, lo que las hace más ligeras computacionalmente y más rápidas de entrenar. Se han escogido GRU frente a LSTM en este caso porque los datos de gestos no requieren una memoria compleja, y las GRU son suficientes para capturar las dependencias temporales con menos recursos computacionales, lo que las hace más adecuadas para manejar datos de video de tamaño moderado.

El siguiente codigo por tanto define un modelo secuencial basado en GRU para clasificar secuencias de características de video. La entrada principal frame_features_input tiene forma (MAX_SEQ_LENGTH, NUM_FEATURES), representando las características temporales de los videos, mientras que mask_input indica qué frames de la secuencia son válidos. La capa Bidirectional envuelve una GRU con 128 unidades, lo que permite procesar las secuencias en ambas direcciones, mejorando la captura de dependencias temporales. Luego, la salida pasa por un promedio global (GlobalAveragePooling1D) que condensa la secuencia en un vector único. Una capa densa de 128 unidades con activación ReLU aprende patrones más complejos, seguida de otra capa Dropout para reducir el sobreajuste. Finalmente, una capa Dense con activación softmax produce las probabilidades para cada clase. El modelo se compila con una pérdida categórica (sparse_categorical_crossentropy), el optimizador Adam y la métrica de precisión (accuracy). Este diseño está optimizado para capturar relaciones temporales en los gestos y proporcionar una clasificación robusta y eficiente.

In [ ]:
LEARNING_RATE = 1e-3  # Tasa de aprendizaje inicial para el optimizador

def get_sequence_model(label_processor):
    # Número de clases a clasificar, derivado del vocabulario del procesador de etiquetas
    num_classes = len(label_processor.get_vocabulary())

    # Entrada para las características de los fotogramas
    frame_features_input = keras.Input((MAX_SEQ_LENGTH, NUM_FEATURES), name="frame_features_input")
    # Entrada para la máscara que indica cuáles fotogramas son válidos
    mask_input = keras.Input((MAX_SEQ_LENGTH,), dtype="bool", name="mask_input")

    # Capa GRU bidireccional para capturar información temporal en ambas direcciones
    x = keras.layers.Bidirectional(
        keras.layers.GRU(
            units=128,  # Número de unidades en la GRU
            return_sequences=True,  # Retorna secuencias completas para la capa siguiente
            kernel_regularizer=keras.regularizers.l2(1e-4)  # Regularización L2 para prevenir sobreajuste
        )
    )(frame_features_input, mask=mask_input)  # Conectar la entrada y usar la máscara
    x = keras.layers.Dropout(0.3)(x)  # Añadir Dropout para evitar el sobreajuste

    # GlobalAveragePooling1D condensa la secuencia en un vector único
    x = keras.layers.GlobalAveragePooling1D()(x)

    # Capa completamente conectada con activación ReLU para aprender características más complejas
    x = keras.layers.Dense(
        128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)
    x = keras.layers.Dropout(0.3)(x)  # Añadir otro Dropout para mayor regularización

    # Capa de salida con activación softmax para clasificar entre las clases
    output = keras.layers.Dense(num_classes, activation="softmax")(x)

    # Definir el modelo con entradas y salida
    model = keras.Model([frame_features_input, mask_input], output)

    # Compilar el modelo con la función de pérdida, optimizador y métrica
    model.compile(
        loss="sparse_categorical_crossentropy",  # Pérdida para clasificación categórica
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),  # Optimizador Adam
        metrics=["accuracy"]  # Métrica de evaluación: exactitud
    )
    return model  # Retornar el modelo compilado

### Entrenamiento

La siguiente función sirve para crear las gráficas de métricas para el entrenamiento y validación del modelo

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    # Recuperar las métricas del historial
    acc = history.history['accuracy']  # Exactitud en entrenamiento
    val_acc = history.history['val_accuracy']  # Exactitud en validación
    loss = history.history['loss']  # Pérdida en entrenamiento
    val_loss = history.history['val_loss']  # Pérdida en validación

    # Crear un rango para las épocas
    epochs = range(1, len(acc) + 1)

    # Gráfico de Accuracy
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, label='Entrenamiento')
    plt.plot(epochs, val_acc, label='Validación')
    plt.title('Accuracy durante el entrenamiento')
    plt.xlabel('Épocas')
    plt.ylabel('Accuracy')
    plt.legend()

    # Gráfico de Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, label='Entrenamiento')
    plt.plot(epochs, val_loss, label='Validación')
    plt.title('Pérdida durante el entrenamiento')
    plt.xlabel('Épocas')
    plt.ylabel('Pérdida')
    plt.legend()

    plt.tight_layout()
    plt.show()

Para el entrenamiento se configura entrenar durante un máximo de 50 épocas con lotes de tamaño 32, ajustándose dinámicamente gracias al callback de EarlyStopping, que detiene el entrenamiento si la pérdida de validación no mejora durante 10 épocas consecutivas, evitando el sobreentrenamiento. La función fit realiza el entrenamiento sobre el conjunto de datos de entrenamiento, evaluando periódicamente el desempeño en el conjunto de validación separado, lo que permite monitorear el ajuste del modelo. Al final del entrenamiento, se evalúan las métricas finales, como la pérdida y la exactitud, tanto en los datos de entrenamiento como de validación.

In [ ]:
EPOCHS = 50  # Número máximo de épocas para entrenar el modelo
BATCH_SIZE = 32  # Tamaño del lote para el entrenamiento

def run_experiment():
    # No necesitas volver a crear el label_processor aquí porque ya es global
    global label_processor

    # Crear el modelo secuencial
    sequence_model = get_sequence_model(label_processor)
    print(sequence_model.summary())

    earlystopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    history = sequence_model.fit(
        [train_data[0], train_data[1]],
        train_labels,
        validation_data=([val_data[0], val_data[1]], val_labels),
        epochs=EPOCHS,
        callbacks=[earlystopping],
        batch_size=BATCH_SIZE
    )

    # Evaluar las métricas del modelo restaurado
    print("\n--- Evaluación del modelo restaurado ---")

    # Evaluar en el conjunto de entrenamiento
    train_metrics = sequence_model.evaluate([train_data[0], train_data[1]], train_labels, verbose=0)
    print(f"- Pérdida en entrenamiento: {train_metrics[0]:.4f}")
    print(f"- Exactitud en entrenamiento: {train_metrics[1]:.4f}")

    # No necesitas acceder a history.validation_data porque ya se usó durante el entrenamiento
    # Las métricas de validación ya están disponibles en history.history
    print(f"- Pérdida en validación: {history.history['val_loss'][-1]:.4f}")  # Último valor de la pérdida de validación
    print(f"- Exactitud en validación: {history.history['val_accuracy'][-1]:.4f}")  # Último valor de la exactitud de validación

    return history, sequence_model

El modelo tiene dos entradas principales: frame_features_input para las características de los fotogramas y mask_input para las máscaras de los fotogramas válidos. La capa bidireccional GRU procesa las características temporales y genera una salida de 256 dimensiones para cada frame. Posteriormente, se aplican capas de Dropout y GlobalAveragePooling1D para reducir el sobreajuste y resumir la información temporal en una sola representación. Finalmente, las capas densas generan una salida de 128 unidades antes de pasar a la capa de clasificación con 64 unidades, que corresponden a las clases del problema. El modelo tiene un total de 1,713,856 parámetros, todos entrenables.

In [ ]:
H, sequence_model = run_experiment()

Los resultados obtenidos muestran un modelo que ha logrado un buen entrenamiento, lo que es notable considerando las limitaciones del conjunto de datos, como el número reducido de muestras y el gran número de clases (64). La exactitud en entrenamiento es del 100%, y la pérdida es baja (0.1026), lo que indica que el modelo ha aprendido a clasificar correctamente las muestras del conjunto de entrenamiento. La exactitud en validación (96.48%) y la pérdida en validación (0.2609) son cercanas a las métricas de entrenamiento, lo que sugiere que el modelo no está sobreajustado y generaliza bien en datos no vistos. Pero veremos acontinuación que este no es un resultado real y que nuestro modelo no generaliza realmente tan bien en conjuntos de datos externos que no ha sido posible de demostrar en validación por la quizas baja cantidad o calidad de datos.

In [ ]:
plot_training_history(H)

### test

### Evaluación en test

Es necesario utilizar la función prepare_all_videos también en el conjunto de prueba porque asegura que los datos de entrada para el modelo sean procesados de manera consistente con los del entrenamiento. Esta función se encarga de extraer características de los videos mediante el extractor preentrenado, recortarlos, redimensionarlos y generar máscaras para manejar la longitud variable de las secuencias. Si los videos del conjunto de prueba no se procesaran de la misma manera, el modelo podría recibir datos en un formato diferente al esperado, lo que afectaría negativamente su desempeño. Además, asegura que las etiquetas estén codificadas de forma coherente con el vocabulario creado durante el entrenamiento, evitando inconsistencias en la clasificación. Este paso garantiza que la evaluación del modelo sea válida y comparativa con el entrenamiento.

In [ ]:
test_data, test_labels = prepare_all_videos_with_augmentation(test_df, test_videos, label_processor, False)

La diferencia en el rendimiento entre el conjunto de validación (96.48%) y el de prueba (72.19%) puede atribuirse a las limitaciones del conjunto de datos utilizado. En primer lugar, el número reducido de muestras por clase (40 en entrenamiento y 10 en prueba) dificulta que el modelo generalice a nuevos datos, especialmente dado el alto número de clases (64). Además, la distribución del conjunto de prueba podría no ser representativa del entrenamiento, lo que genera un desbalance en la capacidad del modelo para reconocer correctamente todas las clases. También es probable que existan variaciones en los videos, como diferencias de iluminación, fondo o calidad, que introducen ruido y dificultan el aprendizaje robusto. Aunque se implementaron técnicas de regularización como Dropout y L2, la alta precisión en validación frente a prueba sugiere un posible sobreajuste a los datos de entrenamiento. Para mejorar, sería esencial ampliar el conjunto de datos, utilizar técnicas avanzadas de data augmentation o incluso recurrir a preentrenamiento en datasets más grandes. Estos factores limitantes explican el rendimiento menor en el conjunto de prueba.

In [ ]:
# -----------------------------------------------------------------------------
# EVALUAR EL MODELO EN TEST
# -----------------------------------------------------------------------------
_, accuracy = sequence_model.evaluate([test_data[0], test_data[1]], test_labels)
print(f"Test accuracy: {round(accuracy * 100, 2)}%")


El siguiente código se encarga de evaluar el rendimiento del modelo en el conjunto de prueba mediante la generación y visualización de una matriz de confusión. Primero, convierte las características y máscaras del conjunto de prueba a tensores para ser compatibles con el modelo entrenado. Luego, realiza predicciones en las muestras de prueba y selecciona la clase con mayor probabilidad para cada muestra (predicted_labels_int). Las etiquetas reales y predichas, que originalmente están en formato codificado, se decodifican a cadenas utilizando el inverted_label_processor para interpretarlas de manera comprensible. Posteriormente, se utiliza la función confusion_matrix de sklearn para calcular la matriz de confusión comparando las etiquetas reales con las predichas, asegurando que se utilicen las clases en el orden del vocabulario del modelo. Finalmente, la matriz de confusión se visualiza con seaborn.heatmap, donde se anotan los valores numéricos y se etiquetan las filas y columnas con las clases correspondientes, permitiendo identificar errores comunes y patrones en las predicciones del modelo.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------------------------------------------------------------
# CREAR EL INVERTED LABEL PROCESSOR
# -----------------------------------------------------------------------------
inverted_label_processor = tf.keras.layers.StringLookup(
    num_oov_indices=0,
    vocabulary=label_processor.get_vocabulary(),
    invert=True
)

# -----------------------------------------------------------------------------
# PREDICCIONES
# -----------------------------------------------------------------------------
# 1) Obtener probabilidades de predicción en el conjunto de prueba
# Convertir los datos de entrada a tensores
test_data_0_tensor = tf.convert_to_tensor(test_data[0])  # Convert to tensor
test_data_1_tensor = tf.convert_to_tensor(test_data[1])  # Convert to tensor

test_probs = sequence_model.predict([test_data_0_tensor, test_data_1_tensor])

# 2) Elegir la clase con mayor probabilidad para cada muestra
predicted_labels_int = np.argmax(test_probs, axis=1)  # Define predicted_labels_int here

# 3) Decodificar etiquetas reales y predichas a cadenas
# Convertir las etiquetas reales a sus cadenas correspondientes
true_labels_str = inverted_label_processor(test_labels).numpy()
# Eliminar la línea que causa el error y aplanar el array
true_labels_str = [label.decode() if isinstance(label, bytes) else label for label in true_labels_str.flatten()]

# Convertir las etiquetas predichas a sus cadenas correspondientes
predicted_labels_str = inverted_label_processor(predicted_labels_int).numpy()
# Eliminar la línea que causa el error y aplanar el array
predicted_labels_str = [label.decode() if isinstance(label, bytes) else label for label in predicted_labels_str.flatten()]

# -----------------------------------------------------------------------------
# MATRIZ DE CONFUSIÓN
# -----------------------------------------------------------------------------
# Obtenemos el vocabulario en el orden definido por el label_processor
vocab = label_processor.get_vocabulary()

# Calcular la matriz de confusión usando sklearn
cm = confusion_matrix(
    y_true=true_labels_str,
    y_pred=predicted_labels_str,
    labels=vocab
)

# Visualización con seaborn
plt.figure(figsize=(20, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=vocab, yticklabels=vocab)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Matriz de confusión - Test")
plt.show()


La matriz de confusión presentada anteriormente muestra cómo el modelo clasifica correctamente (diagonal principal) y cómo comete errores al predecir clases equivocadas (valores fuera de la diagonal) en el conjunto de prueba. Cada fila corresponde a las etiquetas verdaderas, mientras que cada columna representa las predicciones del modelo. En general, se observa una fuerte concentración en la diagonal, lo que indica que el modelo realiza muchas predicciones correctas. Sin embargo, hay errores dispersos, con algunas clases confundidas más frecuentemente, como ocurre entre gestos que podrían ser visualmente similares o estar influenciados por variaciones en la posición de las manos, iluminación o fondo. Esto sugiere que, aunque el modelo tiene un desempeño razonable, ciertas clases tienen características menos distintivas, lo que lleva a confusiones.

A continuación se generará un reporte detallado de las métricas de evaluación del modelo en el conjunto de prueba. El reporte incluirá las métricas de precisión (precision), recuperación (recall), y puntuación F1 (f1-score) para cada clase en el vocabulario, así como el número de muestras por clase (support). Además, mostrará promedios generales, como el promedio macro (macro avg) y el promedio ponderado (weighted avg), junto con la precisión global (accuracy). Este reporte permitirá analizar qué clases tienen un mejor desempeño y cuáles presentan más errores, ayudando a identificar posibles áreas de mejora en el modelo.

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Generar el reporte de clasificación en formato de diccionario
report = classification_report(
    y_true=true_labels_str,
    y_pred=predicted_labels_str,
    target_names=vocab,
    output_dict=True,
    zero_division=0
)

# Convertir el reporte a un DataFrame
report_df = pd.DataFrame(report).transpose()

# Imprimir la tabla en formato legible
print("Reporte de métricas en el conjunto de prueba:")
print(report_df)

Por último visualicemos las metricas obtenidas en una gráfica para una mejor interpretación. La gráfica muestra las métricas de precisión, recuperación y f1-score por clase en el conjunto de prueba, evidenciando un desempeño variable entre clases. Algunas, como "Appear" y "Away", presentan altos valores consistentes en todas las métricas, indicando que el modelo las predice correctamente y captura la mayoría de sus ejemplos. Por otro lado, clases como "Dance" tienen valores cercanos a cero, lo que sugiere que el modelo no logra diferenciarlas adecuadamente. Esto podría deberse a su similitud con otras clases. La discrepancia entre precisión y recuperación en algunas clases, como "Barbecue", señala que aunque el modelo hace predicciones correctas, no identifica todos los ejemplos de estas clases. El f1-score refleja un equilibrio general entre estas métricas, con valores bajos para clases problemáticas. Esta variabilidad sugiere la necesidad de mejorar la calidad del conjunto de datos, empleando técnicas avanzadas para mejorar el reconocimiento de gestos menos representados.

In [ ]:
import matplotlib.pyplot as plt

# Filtrar solo las métricas de las clases (excluir 'accuracy', 'macro avg', etc.)
class_metrics = report_df.iloc[:-3, :]  # Excluir las últimas 3 filas (promedios)

# Graficar Precision, Recall y F1-Score
class_metrics[['precision', 'recall', 'f1-score']].plot(kind='bar', figsize=(15, 7), width=0.8)
plt.title('Métricas por Clase (Precision, Recall, F1-Score)')
plt.xlabel('Clases')
plt.ylabel('Puntuación')
plt.xticks(rotation=90, ha="center")
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

En conclusión, el desarrollo de este modelo para el reconocimiento de gestos en lenguaje de señas ha sido una experiencia que refleja tanto las fortalezas como los desafíos inherentes al problema. El modelo, basado en redes neuronales recurrentes bidireccionales y apoyado por un extractor de características preentrenado como ResNet50, mostró un excelente rendimiento en los conjuntos de entrenamiento y validación, pero una caída en el conjunto de prueba, alcanzando una exactitud del 72.19%. Aunque el conjunto de datos estaba uniformemente distribuido en términos de clases, la cantidad total de ejemplos por clase sigue siendo limitada, lo que podría dificultar al modelo capturar la variabilidad completa de los gestos. La brecha entre los resultados de validación y prueba podría atribuirse a ligeras diferencias en las condiciones de captura de los videos, lo que subraya la importancia de trabajar con datos diversos y representativos. Sin embargo, este trabajo sienta una base sólida y demuestra el potencial de las redes neuronales recurrentes para abordar problemas complejos como este, mostrando un camino claro para futuras mejoras en la calidad y diversidad de los datos o en la optimización de la arquitectura del modelo.